In [1]:
import cv2
import numpy as np
from pathlib import Path
import json
from skimage.feature import blob_log
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
def preprocess_image(img_bgr):
    # 1. Grayscale
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # 2. CLAHE (contrast normalization)
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    norm = clahe.apply(gray)

    # 3. Light Gaussian blur
    blur = cv2.GaussianBlur(norm, (3, 3), 0)

    return blur


In [3]:
# # Paths
# input_dir = Path("./train")
# output_dir = Path("./train_preprocessed")

# output_dir.mkdir(parents=True, exist_ok=True)

# # Image extensions to load
# exts = {".jpg", ".jpeg", ".png"}

# for img_path in input_dir.iterdir():
#     if img_path.suffix.lower() not in exts:
#         continue

#     img = cv2.imread(str(img_path))
#     if img is None:
#         print(f"Failed to load {img_path.name}")
#         continue
    
#     # h, w, c = img.shape
#     # h = h//4
#     # w = w//4
#     processed = preprocess_image(img)
#     # processed = cv2.resize(processed, (w,h))

#     out_path = output_dir / img_path.name
#     cv2.imwrite(str(out_path), processed)

# print("Preprocessing complete.")

In [4]:
# orig = cv2.imread("./train/IMG_20260120_163828_jpg.rf.a10d9d77917aa051664bf105da0d2dc9.jpg")
# orig = cv2.resize(orig, (512,512))
# proc = cv2.imread("./train_preprocessed/IMG_20260120_163828_jpg.rf.a10d9d77917aa051664bf105da0d2dc9.jpg", 0)
# proc = cv2.resize(proc, (512,512))

# cv2.imshow("original", orig)
# cv2.imshow("preprocessed", proc)
# cv2.waitKey(0)
# cv2.destroyAllWindows()

In [5]:
def load_gt_centers(coco_json_path):
    with open(coco_json_path, "r") as f:
        coco = json.load(f)

    # image_id -> list of (x, y)
    gt = {}

    for ann in coco["annotations"]:
        image_id = ann["image_id"]
        x, y, w, h = ann["bbox"]
        cx = x + w / 2
        cy = y + h / 2

        gt.setdefault(image_id, []).append((cx, cy))

    # map image_id -> filename
    id_to_name = {
        img["id"]: img["file_name"]
        for img in coco["images"]
    }

    return gt, id_to_name


In [6]:
def detect_blobs(img_gray,
                 min_sigma=1.5,
                 max_sigma=5.0,
                 threshold=0.02):

    blobs = blob_log(
        img_gray,
        min_sigma=min_sigma,
        max_sigma=max_sigma,
        num_sigma=10,
        threshold=threshold
    )

    # blobs: (y, x, sigma)
    return blobs

In [7]:
def detect_blobs(
    img_gray,
    min_sigma,
    max_sigma,
    threshold,
    scale=0.25
):
    small = cv2.resize(
        img_gray,
        None,
        fx=scale,
        fy=scale,
        interpolation=cv2.INTER_AREA
    )

    blobs = blob_log(
        small,
        min_sigma=min_sigma * scale,
        max_sigma=max_sigma * scale,
        num_sigma=5,
        threshold=threshold
    )

    # Rescale back to original coordinates
    blobs[:, 0] /= scale  # y
    blobs[:, 1] /= scale  # x
    blobs[:, 2] /= scale  # sigma

    return blobs


In [8]:
def visualize_blobs(img_gray, blobs, gt_centers):
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(img_gray, cmap="gray")

    # blobs
    for y, x, sigma in blobs:
        c = plt.Circle((x, y), sigma * 1.5, color="red", fill=False, linewidth=1)
        ax.add_patch(c)

    # GT centers
    for x, y in gt_centers:
        ax.plot(x, y, "go", markersize=3)

    ax.set_title("Red = blobs, Green = GT centers")
    ax.axis("off")
    plt.show()


In [9]:
def compute_recall(blobs, gt_centers, radius_px=4):
    if len(gt_centers) == 0:
        return 1.0

    blob_centers = [(x, y) for y, x, _ in blobs]

    matched = 0
    for gx, gy in gt_centers:
        for bx, by in blob_centers:
            if (gx - bx)**2 + (gy - by)**2 <= radius_px**2:
                matched += 1
                break

    return matched / len(gt_centers)


In [10]:
def evaluate_blob_recall(
    img_dir,
    coco_json,
    min_sigma,
    max_sigma,
    threshold
):
    gt, id_to_name = load_gt_centers(coco_json)

    recalls = []

    for image_id, centers in gt.items():
        img_path = Path(img_dir) / id_to_name[image_id]
        img = cv2.imread(str(img_path), 0)

        blobs = detect_blobs(
            img,
            min_sigma=min_sigma,
            max_sigma=max_sigma,
            threshold=threshold
        )

        r = compute_recall(blobs, centers)
        recalls.append(r)

    return sum(recalls) / len(recalls)

In [11]:
recall = evaluate_blob_recall(
    img_dir="./train_preprocessed",
    coco_json="./train/_annotations.coco.json",
    min_sigma=1.0,
    max_sigma=5.0,
    threshold=0.08
)

print(f"Mean recall: {recall:.3f}")


KeyboardInterrupt: 

In [ ]:
import cv2
import time
from skimage.feature import blob_log

img = cv2.imread(
    "./train_preprocessed/IMG_20260120_164448_jpg.rf.5902e1e625e97cda0addfa514b3c89b5.jpg",
    0
)

small = cv2.resize(
    img,
    None,
    fx=0.25,
    fy=0.25,
    interpolation=cv2.INTER_AREA
)

print("Small image shape:", small.shape)

start = time.time()
blobs = blob_log(
    small,
    min_sigma=1,
    max_sigma=5,
    num_sigma=5,
    threshold=0.08
)
print("Blob count:", len(blobs))
print("Time:", time.time() - start)
